# VGG16-54 R1 R2 — governed fresh epoch-1 smoke

Train/validation only. Owner approval, base source QA and runtime-evidence source QA are hash-pinned to an immutable Git commit. Run all cells once on a fresh NVIDIA L4 runtime. Never upload or open the sealed test.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/CPV301/Traffic_Sign_Detector/M1_D1_Colab')
ARCHIVE = DRIVE / 'cascade_b0_r1.tar.gz'
ARCHIVE_SHA256 = '5a5a9afbf06b9bc2f55546f5a49738ed9ca9ac6f96a78931f78669e71dab4806'
MANIFEST_SHA256 = 'bae147b6f7c6f80ac973923df4b60fd31b6587221fe75eaa2d9ed4b93ee6a51b'
REPO_URL = 'https://github.com/ScottPhantom/Traffic_SIgn_Detection.git'
CHECKOUT_COMMIT = '2e1d981e70e02b19d1998fa17be8c8a0de033df0'
OWNER_APPROVAL = DRIVE / 'vgg16_54_r1_owner_approval.json'
OWNER_APPROVAL_SHA256 = 'f09c7707263b52337957de5dd9ac2bf22712bb9135df8a84880633718ade06af'
SOURCE_QA = DRIVE / 'vgg16_54_r1_source_qa_proof.json'
SOURCE_QA_SHA256 = '10f0ca69e32aed906d0bf8cad36aaacf3d40ea069faec51fb084de0d7ecfb216'
RUNTIME_SOURCE_QA = DRIVE / 'vgg16_54_r1_runtime_r1_source_qa_proof.json'
RUNTIME_SOURCE_QA_SHA256 = 'cbd531302429908354858640ac501e8c0d1805c5dec5462bf5e6a1e611b044f5'
SMOKE_QA = None
SMOKE_QA_SHA256 = None
MODE = 'fresh'
STOP_AFTER = 1
STATE = DRIVE / 'runs/vgg16_54_seed_20261018_r2'
ROOT = Path('/content/vgg16_54_input')
REPO = Path('/content/Traffic_Sign_Detector')


In [ ]:
import hashlib
import subprocess

def sha256(path):
    value = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()

assert len(CHECKOUT_COMMIT) == 40 and all(
    c in '0123456789abcdef' for c in CHECKOUT_COMMIT
)
for path, value in (
    (OWNER_APPROVAL, OWNER_APPROVAL_SHA256),
    (SOURCE_QA, SOURCE_QA_SHA256),
    (RUNTIME_SOURCE_QA, RUNTIME_SOURCE_QA_SHA256),
):
    assert len(value) == 64 and all(c in '0123456789abcdef' for c in value)
    assert sha256(path) == value
assert sha256(ARCHIVE) == ARCHIVE_SHA256
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
import subprocess

assert not REPO.exists(), 'Use a fresh runtime checkout'
subprocess.run(
    ['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)], check=True
)
subprocess.run(
    ['git', 'checkout', '--detach', CHECKOUT_COMMIT], cwd=REPO, check=True
)
observed = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True
).strip()
assert observed == CHECKOUT_COMMIT
subprocess.run(['pip', 'install', '-q', 'uv==0.11.14'], check=True)
subprocess.run(['uv', 'sync', '--frozen'], cwd=REPO / 'detector', check=True)


In [ ]:
import subprocess
import sys

assert not ROOT.exists(), 'Use a fresh extraction destination'
subprocess.run([
    sys.executable, str(REPO / 'detector/colab/extract_cascade_b0_r1.py'),
    str(ARCHIVE), str(ROOT), '--archive-sha256', ARCHIVE_SHA256,
    '--manifest-sha256', MANIFEST_SHA256,
], check=True)


In [ ]:
import subprocess

command = [
    'uv', 'run', 'python', 'colab/run_vgg16_54_r1_r1.py',
    '--root', str(ROOT), '--state', str(STATE),
    '--bundle-manifest-sha256', MANIFEST_SHA256,
    '--owner-approval', str(OWNER_APPROVAL),
    '--owner-approval-sha256', OWNER_APPROVAL_SHA256,
    '--source-qa', str(SOURCE_QA), '--source-qa-sha256', SOURCE_QA_SHA256,
    '--runtime-source-qa', str(RUNTIME_SOURCE_QA),
    '--runtime-source-qa-sha256', RUNTIME_SOURCE_QA_SHA256,
    '--mode', MODE,
]
if STOP_AFTER is not None:
    command += ['--stop-after', str(STOP_AFTER)]
else:
    command += [
        '--smoke-qa', str(SMOKE_QA),
        '--smoke-qa-sha256', SMOKE_QA_SHA256,
    ]
subprocess.run(command, cwd=REPO / 'detector', check=True)


In [ ]:
import json
import sqlite3

execution = json.loads((STATE / 'colab_execution.json').read_text())
print(json.dumps(execution, indent=2, ensure_ascii=False))
with sqlite3.connect(STATE / 'DB_logger.db') as database:
    rows = list(database.execute(
        'SELECT time, stage, event, payload FROM logs '
        'ORDER BY rowid DESC LIMIT 20'
    ))
for row in rows:
    print(row)
attempt = execution['execution_attempts'][0]
assert execution['status'] == 'SMOKE_STOP' and execution['epoch'] == 1
assert execution['sealed_test_accessed'] is False
assert attempt['gpu']['device_name'] == 'NVIDIA L4'
assert len(attempt['runtime_identity']['digest']) == 64
assert attempt['run_binding_sha256'] == attempt['checkpoint_binding_sha256']


## Gate after this notebook

Stop here after epoch 1. Independent artifact QA must verify the R2 checkpoint, runtime/GPU evidence in JSON, SQLite and text log, the binding hashes and `sealed_test_accessed=false`. Resume requires a separately reviewed launcher on a genuinely new L4 runtime. Long training is still forbidden.
